In [0]:
dbutils.widgets.text("status", "")
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("data_ingestion_volume", "")
dbutils.widgets.text("databricks_host", "")
dbutils.widgets.text("job_orchestration_token", "")

# debugging for notbook parameters not being able to recieve from job-1 notebook
kvs = {k: dbutils.widgets.get(k) for k in ["status","catalog","schema","data_ingestion_volume","databricks_host","job_orchestration_token"]}
print("Widget values:", kvs)

# Databricks automatically creates widgets from notebook_params since I sent notebook_parmas from the upstream job I just have to get the params using dbutils
if kvs:
    print(f"Notebook running in job mode the configurations will be dynamically loaded from json file")
    status = dbutils.widgets.get("status")
    catalog = dbutils.widgets.get("catalog")
    schema = dbutils.widgets.get("schema")
    data_ingestion_volume = dbutils.widgets.get("data_ingestion_volume")
    data_ingestion_volume = dbutils.widgets.get("databricks_host")
    data_ingestion_volume = dbutils.widgets.get("job_orchestration_token")

    table_name = dbutils.widgets.get("table_name")
    gold_table_name = dbutils.widgets.get("gold_table_name")
else:
    print(f"Notebook running in interactive mode all the configurations will be fed via hard coded values")
    status = 'success'
    catalog = 'job_orchestration'
    schema = 'default'
    data_ingestion_volume = 'job_orchestration_volume'
    databricks_host = ''
    job_orchestration_token = ''

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.{table_name} AS
            SELECT ct.customer_id, ct.first_name, ct.last_name, ct.phone, ot.order_id, ot.order_date, ot.product, ot.category, ot.quantity, ot.order_amount, ot.state FROM {catalog}.{schema}.customers_table AS ct INNER JOIN {catalog}.{schema}.orders_table AS ot ON ct.customer_id = ot.customer_id ORDER BY customer_id DESC
          """)

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
df = spark.sql(f"""
            SELECT * FROM {catalog}.{schema}.{table_name}
          """)

total_count = df.count()
distinct_count = df.dropDuplicates().count()

has_duplicates = total_count > distinct_count

print(f"has_duplicates ---> {has_duplicates}")

import json

output = {
    "catalog": catalog,
    "schema": schema,
    "data_ingestion_volume":data_ingestion_volume,
    "table_name":table_name,
    "gold_table_name":gold_table_name
}
if kvs:
    dbutils.jobs.taskValues.set(key="metadata", value=json.dumps(output))
    dbutils.jobs.taskValues.set(key="has_duplicates", value=has_duplicates)
    print(f"Notebook running on Job mode sending metadata to another task : {output}")
else:
    print(f"Notebook running on Interactive mode failed to send metadata to another task : {output}")